# Intervention Fork, Replay, and Bundle Workflow

Self-contained: a seeded tiny PyTorch model, no downloads, CPU only. The shape of the workflow is what carries over to a real model.


Capture once, fork the trace, edit each fork, then put the forks in a `Bundle` and compare them
site by site.


In [ ]:
import torch
from torch import nn

import torchlens as tl

SEED = 1604
torch.manual_seed(SEED)
torch.set_grad_enabled(False)


class TinyMLP(nn.Module):
    """Small MLP with two intervention-friendly nonlinearities."""

    def __init__(self) -> None:
        """Initialize layers."""
        super().__init__()
        self.in_proj = nn.Linear(8, 8)
        self.mid_proj = nn.Linear(8, 8)
        self.out_proj = nn.Linear(8, 3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the MLP."""
        x = torch.relu(self.in_proj(x))
        x = torch.relu(self.mid_proj(x))
        return self.out_proj(x)

In [ ]:
model = TinyMLP().eval()
x = torch.randn(4, 8)
clean = tl.trace(model, x)
relu_labels = [site.layer_label for site in clean.find_sites(tl.func("relu"))]
print(relu_labels)

In [ ]:
zero_first = clean.fork("zero_first_relu")
zero_first.do(tl.label(relu_labels[0]), tl.zero_ablate())

scale_second = clean.fork("scale_second_relu")
scale_second.do(tl.label(relu_labels[1]), tl.scale(0.5))

bundle = tl.Bundle(
    {"clean": clean, "zero_first": zero_first, "scale_second": scale_second}, baseline="clean"
)
print(bundle.names)

In [ ]:
output_norms = bundle.map(lambda member: float(torch.linalg.vector_norm(member.output_ops[-1].out)))
for name, value in output_norms.items():
    print(f"{name:16s} output norm {value:.4f}")

comparison = bundle.at(tl.label(relu_labels[0])).compare()  # cosine distance, member by member
print(bundle.names)
print(comparison)

The bundle keeps branch names and common sites together, so metrics can be computed across
members without aligning layer labels by hand.
